Build a Clean Dataset

This notebook demonstrates building the gold-layer dataset from clean silver data. It produces a **one-row-per-piece** parquet file with cumulative travel times, inter-stage partial times, and production context — segmented by die matrix.

### What this notebook does

1. Load clean pieces from `silver.clean_pieces` (all cleaning already applied)
2. Verify data quality: no zeros, no outliers, monotonic order, valid OEE
3. Compute partial times between process stages
4. Mark production gaps and assign run IDs
5. Inspect the final dataset structure per die matrix
6. Export to parquet (locally, or S3 when on AWS)

In [3]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path

for line in open("../infra/.env"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k] = v


u    = os.environ["POSTGRES_USER"]
pw   = os.environ["POSTGRES_PASSWORD"]
db   = os.environ["POSTGRES_DB"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]

engine = create_engine(f"postgresql://{u}:{pw}@{host}:{port}/{db}")
print("Connected to database.")


Connected to database.


## 1. Load clean data from silver

The silver table contains only valid pieces — all signal-level noise, tracking failures, outliers, and monotonic violations were removed by the `01_bronze_to_silver` notebook.

In [5]:
silver = pd.read_sql("""
    SELECT
        timestamp, piece_id, die_matrix,
        lifetime_2nd_strike_s,
        lifetime_3rd_strike_s,
        lifetime_4th_strike_s,
        lifetime_auxiliary_press_s,
        lifetime_bath_s,
        lifetime_general_s,
        oee_cycle_time_s
    FROM silver.clean_pieces
    ORDER BY timestamp
""", engine)


# oee_cycle_time_s is stored as text in the DB — coerce to float
silver["oee_cycle_time_s"] = pd.to_numeric(silver["oee_cycle_time_s"], errors="coerce")

print(f"Loaded {len(silver):,} pieces from silver.clean_pieces")
print(f"Date range: {silver['timestamp'].min()} → {silver['timestamp'].max()}")
print(f"Die matrices: {sorted(silver['die_matrix'].unique())}")
silver.head()


Loaded 140,404 pieces from silver.clean_pieces
Date range: 2025-11-06 15:25:16.426000+00:00 → 2026-02-26 00:57:01.946000+00:00
Die matrices: [np.int64(4974), np.int64(5052), np.int64(5090), np.int64(5091)]


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,oee_cycle_time_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,NaN
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,NaN
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,NaN
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,NaN
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,NaN


## 2. Verify data quality

Quick sanity check that the silver data is indeed clean — no zeros, no extreme values, monotonic order holds.

In [6]:
lifetime_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]

print("=" * 50)
print("QUALITY GATE")
print("=" * 50)

# --- Check 1: No zeros ---
zero_counts = (silver[lifetime_cols] == 0).sum()
print("\n[1] Zero values per lifetime column:")
print(zero_counts.to_string())
assert zero_counts.sum() == 0, "FAIL: zero values found — cleaning incomplete!"
print("    ✓ No zeros\n")

# --- Check 2: No nulls in key identification columns ---
null_counts = silver[["piece_id", "die_matrix"] + lifetime_cols].isnull().sum()
print("[2] Null values per column:")
print(null_counts.to_string())
assert null_counts[["piece_id", "die_matrix"]].sum() == 0, "FAIL: null piece_id or die_matrix!"
print("    ✓ No nulls in key columns\n")

# --- Check 3: Monotonic order: 2nd < 3rd < 4th < aux < bath ---
violations = (
    (silver["lifetime_3rd_strike_s"]      <= silver["lifetime_2nd_strike_s"]) |
    (silver["lifetime_4th_strike_s"]      <= silver["lifetime_3rd_strike_s"]) |
    (silver["lifetime_auxiliary_press_s"] <= silver["lifetime_4th_strike_s"]) |
    (silver["lifetime_bath_s"]            <= silver["lifetime_auxiliary_press_s"])
).sum()
print(f"[3] Monotonic-order violations: {violations}")
print("    ✓ Cumulative times increase monotonically\n")

# --- Check 4: OEE within expected range (11–16 s) or NULL ---
oee_valid    = silver["oee_cycle_time_s"].dropna()
out_of_range = ((oee_valid < 11) | (oee_valid > 16)).sum()
null_oee     = silver["oee_cycle_time_s"].isnull().sum()
print(f"[4] OEE out of range (11–16 s): {out_of_range}")
print(f"    OEE null (expected for first pieces in each run): {null_oee:,}")
print("    ✓ OEE quality check passed\n")

print("All quality checks passed ✓")


QUALITY GATE

[1] Zero values per lifetime column:
lifetime_2nd_strike_s         0
lifetime_3rd_strike_s         0
lifetime_4th_strike_s         0
lifetime_auxiliary_press_s    0
lifetime_bath_s               0
    ✓ No zeros

[2] Null values per column:
piece_id                      0
die_matrix                    0
lifetime_2nd_strike_s         0
lifetime_3rd_strike_s         0
lifetime_4th_strike_s         0
lifetime_auxiliary_press_s    0
lifetime_bath_s               0
    ✓ No nulls in key columns

[3] Monotonic-order violations: 0
    ✓ Cumulative times increase monotonically

[4] OEE out of range (11–16 s): 0
    OEE null (expected for first pieces in each run): 34,853
    ✓ OEE quality check passed

All quality checks passed ✓


## 3. Dataset overview per die matrix

Each die matrix has different tooling geometry and expected travel times. All analysis must be segmented by matrix.

In [7]:
overview = silver.groupby("die_matrix").agg(
    pieces        = ("piece_id",        "count"),
    median_bath_s = ("lifetime_bath_s", "median"),
    p10_bath_s    = ("lifetime_bath_s", lambda x: x.quantile(0.10)),
    p90_bath_s    = ("lifetime_bath_s", lambda x: x.quantile(0.90)),
    start_date    = ("timestamp",       "min"),
    end_date      = ("timestamp",       "max"),
).reset_index()

overview["active_days"] = (overview["end_date"] - overview["start_date"]).dt.days

print("Per-matrix overview:")
print(overview.to_string(index=False))


Per-matrix overview:
 die_matrix  pieces  median_bath_s  p10_bath_s  p90_bath_s                       start_date                         end_date  active_days
       4974   15531      56.000000   54.700001   59.000000 2025-11-13 20:19:47.883000+00:00 2025-11-25 09:25:29.704000+00:00           11
       5052   20887      58.299999   55.500000   61.799999 2025-11-06 15:25:16.426000+00:00 2026-02-25 03:45:42.232000+00:00          110
       5090   81677      58.099998   55.599998   62.900002 2025-12-04 21:06:17.707000+00:00 2026-02-17 13:31:14.137000+00:00           74
       5091   22309      56.799999   55.200001   61.700001 2025-11-25 18:37:19.931000+00:00 2026-02-26 00:57:01.946000+00:00           92


## 4. Compute partial times between stages

Since lifetime columns are cumulative from furnace exit, the time spent **between two consecutive stages** is computed by subtraction. All values in **seconds**.

| Partial column | Calculation | What it measures |
|---|---|---|
| `partial_furnace_to_2nd_strike_s` | `lifetime_2nd_strike_s` | Robot pick, transfer, positioning at main press |
| `partial_2nd_to_3rd_strike_s` | `lifetime_3rd - lifetime_2nd` | Press retraction, repositioning between strikes |
| `partial_3rd_to_4th_strike_s` | `lifetime_4th - lifetime_3rd` | Transfer to drill station on main press |
| `partial_4th_strike_to_auxiliary_press_s` | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| `partial_auxiliary_press_to_bath_s` | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

In [8]:
gold = silver.copy()

# Partial times: time spent between consecutive process stages (all in seconds)
gold["partial_furnace_to_2nd_s"] = gold["lifetime_2nd_strike_s"]
gold["partial_2nd_to_3rd_s"]     = gold["lifetime_3rd_strike_s"]      - gold["lifetime_2nd_strike_s"]
gold["partial_3rd_to_4th_s"]     = gold["lifetime_4th_strike_s"]      - gold["lifetime_3rd_strike_s"]
gold["partial_4th_to_aux_s"]     = gold["lifetime_auxiliary_press_s"] - gold["lifetime_4th_strike_s"]
gold["partial_aux_to_bath_s"]    = gold["lifetime_bath_s"]            - gold["lifetime_auxiliary_press_s"]

partial_cols = [
    "partial_furnace_to_2nd_s",
    "partial_2nd_to_3rd_s",
    "partial_3rd_to_4th_s",
    "partial_aux_to_bath_s",
]

# All partials must be strictly positive (monotonic order already verified)
neg = (gold[partial_cols] < 0).sum()
assert neg.sum() == 0, f"Negative partial times found:\n{neg}"
print("✓ All partial times are positive\n")

print("Partial time statistics (seconds):")
gold[partial_cols].describe().round(2)


✓ All partial times are positive

Partial time statistics (seconds):


,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_aux_to_bath_s
count,140404.00,140404.00,140404.00,140404.00
mean,18.46,6.88,13.87,1.64
std,2.28,0.68,1.07,0.09
min,9.40,6.20,12.60,1.40
25%,16.80,6.60,13.40,1.60
50%,17.80,6.80,13.70,1.60
75%,19.60,7.00,14.00,1.70
max,31.00,23.20,31.80,18.20


## 5. Partial times per die matrix

Each matrix has its own expected timing profile. These medians serve as the **reference behavior** for deviation detection.

In [9]:
partial_by_matrix = gold.groupby("die_matrix")[partial_cols].median().round(2)

# Sanity: sum of partials should equal median bath time
partial_by_matrix["sum_partials"] = partial_by_matrix[partial_cols].sum(axis=1)
ref_bath = gold.groupby("die_matrix")["lifetime_bath_s"].median().round(2).rename("median_bath_s")

result = partial_by_matrix.join(ref_bath)
print("Median partial times per die matrix (seconds):")
print(result.to_string())


Median partial times per die matrix (seconds):
            partial_furnace_to_2nd_s  partial_2nd_to_3rd_s  partial_3rd_to_4th_s  partial_aux_to_bath_s  sum_partials  median_bath_s
die_matrix                                                                                                                          
4974                            17.3                   6.5                  13.1                    1.8          38.7           56.0
5052                            18.3                   6.9                  13.7                    1.6          40.5           58.3
5090                            17.7                   6.8                  13.8                    1.6          39.9           58.1
5091                            17.9                   6.6                  13.5                    1.6          39.6           56.8


## 6. Mark production gaps

Flag pieces that follow a production gap (> 5 minutes). Assign a `production_run_id` to group consecutive pieces within the same uninterrupted run.

In [10]:
gold = gold.sort_values("timestamp").reset_index(drop=True)

# Time gap since the previous piece (in seconds)
gold["gap_s"] = gold["timestamp"].diff().dt.total_seconds()
gold["production_gap"] = gold["gap_s"] > 300  # 5 minutes
gold["production_run_id"] = gold["production_gap"].cumsum().astype(int)

n_gaps = gold["production_gap"].sum()
n_runs = gold["production_run_id"].nunique()
print(f"Total pieces:             {len(gold):,}")
print(f"Production gaps (>5 min): {n_gaps:,}")
print(f"Production runs:          {n_runs}")

gap_minutes = gold.loc[gold["production_gap"], "gap_s"] / 60
print(f"\nGap duration stats (minutes):")
print(gap_minutes.describe().round(1))


Total pieces:             140,404
Production gaps (>5 min): 775
Production runs:          776

Gap duration stats (minutes):
count      775.0
mean       157.7
std       1000.4
min          5.0
25%          7.6
50%         12.4
75%         26.9
max      21216.6
Name: gap_s, dtype: float64


## 7. Final dataset structure

The gold dataset has one row per piece with all identification, cumulative times, partial times, OEE, and production context.

In [11]:
ordered_cols = [
    # Identification
    "timestamp", "piece_id", "die_matrix",
    # Cumulative times (from furnace exit)
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    # Partial (inter-stage) times
    "partial_furnace_to_2nd_s",
    "partial_2nd_to_3rd_s",
    "partial_3rd_to_4th_s",
    "partial_4th_to_aux_s",
    # Production context
    "oee_cycle_time_s",
    "production_gap",
    "production_run_id",
]

gold_final = gold[ordered_cols].copy()

print(f"Gold dataset: {gold_final.shape[0]:,} rows × {gold_final.shape[1]} columns")
print("\nColumn types:")
print(gold_final.dtypes.to_string())
print()
gold_final.head()


Gold dataset: 140,404 rows × 16 columns

Column types:
timestamp                     datetime64[ns, UTC]
piece_id                                   object
die_matrix                                  int64
lifetime_2nd_strike_s                     float64
lifetime_3rd_strike_s                     float64
lifetime_4th_strike_s                     float64
lifetime_auxiliary_press_s                float64
lifetime_bath_s                           float64
lifetime_general_s                        float64
partial_furnace_to_2nd_s                  float64
partial_2nd_to_3rd_s                      float64
partial_3rd_to_4th_s                      float64
partial_4th_to_aux_s                      float64
oee_cycle_time_s                          float64
production_gap                               bool
production_run_id                           int64



,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_s,partial_2nd_to_3rd_s,partial_3rd_to_4th_s,partial_4th_to_aux_s,oee_cycle_time_s,production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,NaN,False,0
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,NaN,False,0
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,NaN,False,0
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,NaN,False,0
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,NaN,False,0


## 8. Export to parquet

Save the gold dataset. When running on AWS, change `GOLD_DIR` to an S3 path.

In [12]:
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)
out_path = GOLD_DIR / "pieces.parquet"

gold_final.to_parquet(out_path, index=False)
print(f"Exported: {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

# --- Round-trip validation ---
loaded = pd.read_parquet(out_path)
assert loaded.shape == gold_final.shape, \
    f"Shape mismatch: written {gold_final.shape} vs read {loaded.shape}"
assert list(loaded.columns) == list(gold_final.columns), \
    "Column order mismatch after reload"

print(f"\n✓ Round-trip validated")
print(f"  Rows:    {loaded.shape[0]:,}")
print(f"  Columns: {loaded.shape[1]}")
print(f"  Columns: {list(loaded.columns)}")


Exported: ../data/gold/pieces.parquet
File size: 4.0 MB

✓ Round-trip validated
  Rows:    140,404
  Columns: 16
  Columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'partial_furnace_to_2nd_s', 'partial_2nd_to_3rd_s', 'partial_3rd_to_4th_s', 'partial_4th_to_aux_s', 'oee_cycle_time_s', 'production_gap', 'production_run_id']
